In [ ]:
%%capture
!pip install --upgrade scikit-learn
!pip install --upgrade gensim
!apt-get install git

In [ ]:
!git clone https://github.com/YJiangcm/SST-2-sentiment-analysis.git

Cloning into 'SST-2-sentiment-analysis'...
remote: Enumerating objects: 85, done.
remote: Counting objects: 100% (85/85), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 85 (delta 44), reused 29 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (85/85), 478.79 KiB | 6.06 MiB/s, done.
Resolving deltas: 100% (44/44), done.


In [ ]:
from google.colab import files
import pandas as pd
import re
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD

from sklearn.model_selection import train_test_split, cross_val_score

from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, classification_report, accuracy_score, f1_score, ConfusionMatrixDisplay
from sklearn import metrics

import gensim
import gensim.downloader

def Preprocess(Text):
    msg = re.sub('[^a-zA-Z]', ' ', Text)
    msg = msg.lower()
    msg = msg.split()
    msg = ' '.join(msg)
    return msg

In [ ]:
W2V = gensim.downloader.load('word2vec-google-news-300')
FTX = gensim.downloader.load('fasttext-wiki-news-subwords-300')

[==================================================] 100.0% 1662.8/1662.8MB downloaded
[==================================================] 100.0% 958.5/958.4MB downloaded


In [ ]:
trainset = pd.read_csv("SST-2-sentiment-analysis/data/train.tsv", sep='\t', header=None, names=["class", "review"])
testset = pd.read_csv("SST-2-sentiment-analysis/data/test.tsv", sep='\t', header=None, names=["class", "review"])
trainset["p-review"] = trainset["review"].apply(Preprocess)
testset["p-review"] = testset["review"].apply(Preprocess)


In [ ]:
ntrain = len(trainset)
ntest = len(testset)
print(ntrain, ntest)

corpus = np.array(trainset["p-review"], testset["p-review"])
cvect = CountVectorizer(stop_words="english", min_df=5)
cvectors = cvect.fit_transform(corpus).toarray()

idfvect = TfidfVectorizer(stop_words="english", min_df=5)
idfvectors = idfvect.fit_transform(corpus).toarray()

svdvect = TruncatedSVD(n_components=1024)
svdvectors = svdvect.fit_transform(idfvectors)

vocab = cvect.get_feature_names_out()

w2vvectors = []
for i in range(len(cvectors)):
  x = np.zeros(300)
  for idx in range(len(vocab)):
    if cvectors[i][idx] == 0:
      continue
    if W2V.has_index_for(vocab[idx]):
      np.add(x, W2V[vocab[idx]] * cvectors[i][idx], x)
  w2vvectors.append(x)

ftxvectors = []
for i in range(len(cvectors)):
  x = np.zeros(300)
  for idx in range(len(vocab)):
    if cvectors[i][idx] == 0:
      continue
    if FTX.has_index_for(vocab[idx]):
      np.add(x, FTX[vocab[idx]] * cvectors[i][idx], x)
  ftxvectors.append(x)

label_encoder = LabelEncoder()
classes = np.array(trainset["class"], testset["class"])

CTrain = cvectors[:ntrain]
CTest = cvectors[-ntest:]

TrainCls = classes[:ntrain]
TestCls = classes[-ntest:]

IdfTrain = idfvectors[:ntrain]
IdfTest = idfvectors[-ntest:]

SvdTrain = svdvectors[:ntrain]
SvdTest = svdvectors[-ntest:]

W2vTrain = w2vvectors[:ntrain]
W2vTest = w2vvectors[-ntest:]

FtxTrain = ftxvectors[:ntrain]
FtxTest = ftxvectors[-ntest:]

score = "accuracy"
print(len(CTrain[0]), len(IdfTrain[0]), len(SvdTrain[0]))

6920 1821
2680 2680 1024


Hiệu quả phân lớp khi sử dụng Count Vectorizer

In [ ]:
cls = MultinomialNB()
cls.fit(CTrain, TrainCls)
cv_score = cross_val_score(cls, CTest, TestCls, scoring=score, cv=10)
print("MultinomialNB", cv_score.mean())

cls = LogisticRegression()
cls.fit(CTrain, TrainCls)
cv_score = cross_val_score(cls, CTest, TestCls, scoring=score, cv=10)
print("Logistic Regression", cv_score.mean())

MultinomialNB 0.7193748874076744
Logistic Regression 0.7062060889929743


Hiệu quả khi sử dụng TfIdf Vectorizer

In [ ]:
cls = MultinomialNB()
cls.fit(IdfTrain, TrainCls)
cv_score = cross_val_score(cls, IdfTest, TestCls, scoring=score, cv=10)
print("MultinomialNB", cv_score.mean())

cls = LogisticRegression()
cls.fit(IdfTrain, TrainCls)
cv_score = cross_val_score(cls, IdfTest, TestCls, scoring=score, cv=10)
print("Logistic Regression", cv_score.mean())

MultinomialNB 0.7232180387918093
Logistic Regression 0.7210322464420824


Hiệu quả khi sử dụng SVD

In [ ]:
cls = LogisticRegression()
cls.fit(SvdTrain, TrainCls)
cv_score = cross_val_score(cls, SvdTest, TestCls, scoring=score, cv=10)
print("Logistic Regression", cv_score.mean())

Logistic Regression 0.7287215516723714


Hiệu quả khi sử dụng Word-to-Vec

In [ ]:
cls = LogisticRegression()
cls.fit(W2vTrain, TrainCls)
cv_score = cross_val_score(cls, W2vTest, TestCls, scoring=score, cv=10)
print("Logistic Regression", cv_score.mean())

Logistic Regression 0.7391250825677055


Hiệu quả khi sử dụng Fasttext

In [ ]:
cls = LogisticRegression()
cls.fit(FtxTrain, TrainCls)
cv_score = cross_val_score(cls, FtxTest, TestCls, scoring=score, cv=10)
print("Logistic Regression", cv_score.mean())

Logistic Regression 0.7594667627454512
